# Drug-Target Interaction Predictor — GPU Training (Colab)

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run All Cells in order
3. When Step 2 says "Upload file", pick `davis_full.csv` from your laptop
4. **Download `best_model.pt` when prompted**

In [ ]:
# Step 0: Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}')
    print(f'VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('No GPU! Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Step 1: Clone repo
import os
!git clone https://github.com/marine9056/drug-target-Predictor.git
%cd drug-target-Predictor
print('Repo ready:', os.listdir('.'))

In [ ]:
# Step 2: Upload davis_full.csv from your laptop
# A "Choose Files" button will appear — click it and select davis_full.csv
# File location on your laptop: data/raw/davis_full.csv
from google.colab import files
import shutil

os.makedirs('data/raw', exist_ok=True)
print('Click "Choose Files" and select davis_full.csv from your laptop:')
uploaded = files.upload()

for filename in uploaded:
    shutil.move(filename, f'data/raw/{filename}')
    size_mb = os.path.getsize(f'data/raw/{filename}') / 1e6
    print(f'Uploaded: {filename} ({size_mb:.1f} MB) -> data/raw/{filename}')

In [ ]:
# Step 3: Verify data
import pandas as pd
df = pd.read_csv('data/raw/davis_full.csv')
print(f'Dataset: {len(df)} rows')
print(f'pKd range: {df.iloc[:, -1].min():.2f} - {df.iloc[:, -1].max():.2f}')
print(f'Unique drugs: {df.iloc[:, 0].nunique()}, proteins: {df.iloc[:, 1].nunique()}')

In [ ]:
# Step 4: Install dependencies
!pip install -q torch torch-geometric rdkit pyyaml requests pandas numpy scipy scikit-learn

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# Step 5: Run GPU training (~15 min on T4)
!python train_gpu.py

In [ ]:
# Step 6: Show results
checkpoint_dir = 'models/checkpoints'
print('Checkpoint files:')
for f in sorted(os.listdir(checkpoint_dir)):
    if f.endswith('.pt'):
        size_mb = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
        print(f'  {f} ({size_mb:.1f} MB)')

if os.path.exists('outputs/gpu_results.txt'):
    print('\n--- Test Set Results ---')
    print(open('outputs/gpu_results.txt').read())

In [ ]:
# Step 7: DOWNLOAD best_model.pt — CRITICAL, session will expire!
from google.colab import files

checkpoint_path = 'models/checkpoints/best_model.pt'
if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / 1e6
    print(f'Downloading best_model.pt ({size_mb:.1f} MB)...')
    print('Save to: models/checkpoints/best_model.pt in your local repo')
    files.download(checkpoint_path)
else:
    print('ERROR: best_model.pt not found! Training may have failed.')